<a href="https://colab.research.google.com/github/JuanFelipeJL/Modelo-Predictivo/blob/main/Calidad_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calidad de datos

##Dimensiones de calidad por trabajar:


*   Completitud

*   Unicidad

*   Válidez
*   Consistencia

*   Integridad


*   Actualidad



**Objetivo:** Diagnosticar las 6 dimensiones de calidad de datos, limpiar y preparar del dataset para el modelo

In [ ]:
%%capture
!pip install missingno ydata_profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from ydata_profiling import ProfileReport
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 60)
sns.set_style('whitegrid')

print("Librerias cargadas correctamente")

In [ ]:
#Cargarmos dataset original sin modificaciones
df = pd.read_csv("telco_customer_data_v2.csv")
print(f"Shape: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


# 1. Perfilado de datos (Reporte HTML)

Generamos reporte automáticp con ydata-profiling sobre los datos originales sin ninguna modificación previa. Este reporte es la evidencia del estado inicial del dataset

In [ ]:
# Generar el reporte de perfilado sobre datos ORIGINALES
profile = ProfileReport(
    df,
    title="Telco Costumer - Perfilado Datos Originales",
    explorative=True
)

# Exportar a HTML
profile.to_file("TelcoCostumer.html")
print("Reporte HTML exportado: TelcoCostumer.html")

In [ ]:
# Vista rapida del estado original
print("="*60)
print("RESUMEN DEL DATASET ORIGINAL")
print("="*60)
print(f"\nFilas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Celdas totales: {df.shape[0] * df.shape[1]:,}")
print(f"Celdas con nulos: {df.isnull().sum().sum():,}")
print(f"Porcentaje de nulos total: {df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.2f}%")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())

# 2. Diagnositoco de las 6 dimensiones de calidad

### 2.1 Completitud
¿Que tan complejo es el dataset?¿Qué porciones de valores falta por columna?

In [ ]:
# Conteo y porcentaje de nulos por columna
nulos = df.isnull().sum()
pct_nulos = (df.isnull().mean() * 100).round(2)
completitud = (100 - pct_nulos).round(2)

resumen_completitud = pd.DataFrame({
    'Nulos': nulos,
    '% Nulos': pct_nulos,
    '% Completitud': completitud
}).sort_values('% Nulos', ascending=False)

print("COMPLETITUD POR COLUMNA")
print("="*50)
print(resumen_completitud.to_string())
print(f"\nCompletitud general del dataset: {(1 - df.isnull().sum().sum() / (df.shape[0]*df.shape[1]))*100:.2f}%")

In [ ]:
# Visualizacion de valores faltantes con missingno
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matriz de nulidad
plt.sca(axes[0])
msno.matrix(df, ax=axes[0], fontsize=8, sparkline=False)
axes[0].set_title('Matriz de Nulidad', fontsize=12, fontweight='bold')

# Barras de completitud
plt.sca(axes[1])
msno.bar(df, ax=axes[1], fontsize=8)
axes[1].set_title('Completitud por Columna', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Dendrograma de correlacion de nulidad: columnas que faltan juntas
fig, ax = plt.subplots(figsize=(12, 4))
msno.dendrogram(df, ax=ax, fontsize=9)
ax.set_title('Dendrograma de Correlacion de Nulidad\n(columnas que faltan juntas se agrupan)',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Diagnostico de completitud


*   **PaymentMethod** tiene 5.10% de nulos — La columna con más nulos
*   **Dependents, Partner y OnlineSecurity** tienen entre 4.17% y 5.09% de nulos — forman un grupo con ausencias moderadas similares


*   **DeviceProtection, StreamingTV, StreamingMovies, OnlineBackup y TechSupport** tienen entre 3.90% y 4.13% de nulos — patrón correlacionado (servicios adicionales que faltan juntos, como se confirma en el dendrograma)
*  **MultipleLines** tiene 2.67% de nulos — nivel intermedio-bajo de nulos


*   **TotalCharges** tiene 1.52% de nulos — A pesar de no parecer tan alto es significativo por ser variable financiera clave, en este caso, variable objetivo, Es una variable crítica, por lo que ese pequeño grupo requerirá atención.
*   **gender, SeniorCitizen, tenure y MonthlyCharges** tienen entre 0.55% y 1.07% de nulos — son niveles bajos con poco impacto


*  Las columnas **customerID, InternetService, PhoneService, Contract, PaperlessBilling y Churn** están 100% completas — No requieren de un tratamiento
*   **Completitud general:** 97.76% — el dataset tiene buena calidad global, la mayoría de los datos están listos para trabajarse, pero las columnas de servicios adicionales presentan un patrón de ausencia correlacionada que debe tratarse de forma conjunta









###2.2 Unicidad
No hay filas duplicadas o indentificadores repetidos?

In [ ]:
# Filas completamente duplicadas
n_dupes = df.duplicated().sum()
print(f"Filas completamente duplicadas: {n_dupes}")

if n_dupes > 0:
    print(f"\nFilas duplicadas encontradas:")
    display(df[df.duplicated(keep=False)].sort_values(by=['customerID']))

# Verificar si customerID es el identificador único
n_ids = df['customerID'].nunique()
print(f"\nIDs únicos: {n_ids}, de {len(df)} filas")
print(f"Porcentaje de unicidad (customerID): {n_ids/len(df)*100:.2f}%")

# Un cliente puede aparecer múltiples veces — verificamos
id_counts = df['customerID'].value_counts()
id_counts = id_counts[id_counts > 1]
if len(id_counts) > 0:
    print(f"\nCustomerIDs que aparecen más de 1 vez: {len(id_counts)}")
    print(f"ID más repetido: '{id_counts.index[0]}' con {id_counts.iloc[0]} apariciones")
    print(f"\nTop 10 IDs más repetidos:")
    print(id_counts.head(10))
else:
    print("\nTodos los customerID son únicos — sin duplicados por identificador")

**Diagnostico de unicidad:**

Custumer ID a pesar de no ser tan importante en la etapa de predicción, en la etapa de calidad sí lo es, podemos verificar con esta variable que no hayan valores duplicados, para este caso se encontro lo siguiente:

*   No hay filas duplicadas
*   Cada customerID es único: 70,000 IDs únicos para 70,000 filas


*   El dataset tiene un identificador transaccional claro (customerID), a diferencia de otros datasets donde no existe una clave primaria explícita
*   Esto es consistente con la naturaleza del dataset: cada fila representa un cliente diferente, no hay re-entradas ni registros repetidos


*   El porcentaje de unicidad es del 100%, por lo que no se requiere ninguna acción de limpieza por duplicados






###2.3 Validez
¿Los tipos de datos son correctos? ¿Los valores caen dentro de rangos lógicos?


In [ ]:
#Conocer tipos de datos del dataset
print(df.dtypes)

In [ ]:
# Tipos de datos actuales vs esperados
tipos_actuales = df.dtypes
tipos_esperados = {
    'customerID': 'object (OK)',
    'gender': 'object (OK)',
    'SeniorCitizen': 'object (FALLA — debería ser int o bool)',
    'Partner': 'object (OK)',
    'Dependents': 'object (OK)',
    'tenure': 'float64 (FALLA — debería ser int)',
    'PhoneService': 'object (OK)',
    'MultipleLines': 'object (OK)',
    'InternetService': 'object (OK)',
    'OnlineSecurity': 'object (OK)',
    'OnlineBackup': 'object (OK)',
    'DeviceProtection': 'object (OK)',
    'TechSupport': 'object (OK)',
    'StreamingTV': 'object (OK)',
    'StreamingMovies': 'object (OK)',
    'Contract': 'object (OK)',
    'PaperlessBilling': 'object (OK)',
    'PaymentMethod': 'object (OK)',
    'MonthlyCharges': 'float64 (OK)',
    'TotalCharges': 'object (FALLA — debería ser float64)',
    'Churn': 'object (OK)',
}

validez_tipos = pd.DataFrame({
    'Tipo Actual': tipos_actuales,
    'Evaluacion': tipos_esperados
})
print("VALIDEZ DE TIPOS")
print("-"*70)
print(validez_tipos.to_string())

In [ ]:
# Validez de RANGOS para variables numericas
print("VALIDEZ DE RANGOS")
print("="*70)

# TotalCharges debe convertirse primero a numérico
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in numericas:
    s = df[col].dropna()
    problemas = []

    if s.min() < 0:
        problemas.append(f"valores negativos (min={s.min()})")

    if col == 'tenure' and s.max() > 72:
        problemas.append(f"max={s.max():.0f} meses (sospechoso)")

    if col == 'tenure' and (s == 0).sum() > 0:
        n_zeros = (s == 0).sum()
        problemas.append(f"{n_zeros} valores en 0 (clientes recién ingresados, revisar)")

    if col == 'MonthlyCharges' and s.max() > 200:
        problemas.append(f"max={s.max():.1f} USD (sospechoso)")

    if col == 'TotalCharges' and s.max() > 10000:
        problemas.append(f"max={s.max():.1f} USD (revisar)")

    status = ', '.join(problemas) if problemas else "OK"
    print(f"  {col:20s} | min={s.min():>10.1f} | max={s.max():>10.1f} | {status}")

In [ ]:
# Verificar categoricas
print("VALIDEZ DE CATEGORICAS")
print("="*50)

#Incluimos todos los tipo object así no sea el tipo de dato correcto para esa variable con el fin de terminar de validar
categoricas = ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
               'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
               'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
               'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
               'TotalCharges', 'Churn']

for col in categoricas:
    vals = df[col].dropna().unique()
    print(f"\n{col} ({len(vals)} categorias): {sorted(vals)}")

**Diagnóstico de validez:**



*   **customerID** tiene 70,000 categorías únicas — confirmamos que es clave primaria sin repeticiones

*   **SeniorCitizen** tiene 5 categorías inconsistentes ('0', '1', 'No', 'Yes', 'not senior') — mezcla de formatos numérico y texto que debe unificarse antes de convertir a int o bool

*   **tenure** es float64 por los nulos, debería ser int — tiene valores negativos (min=-10) y máximo de 999 meses que son errores de entrada

*   **TotalCharges** llegó como object y al convertir a numérico muestra valores negativos (min=-99.12) y máximo de 1,244,434 imposible — además ya aparece procesada como np.float64 lo que confirma que contenía caracteres no numéricos
*   **MonthlyCharges** tiene un máximo de 1,499.8 USD sospechoso para telecomunicaciones


*   **gender** tiene 7 categorías cuando debería tener 2 — 'FEMALE', 'Female', 'f' y 'Male', 'Man', 'm', 'male' deben unificarse


*   **OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies** tienen 5 categorías cuando deberían tener 3 — 'True' y 'Y' son variantes no estándar de 'Yes'


*   **Contract** tiene 4 categorías cuando debería tener 3 — 'M-M' es variante incorrecta de 'Month-to-month'


*   **PaymentMethod** tiene 5 categorías — 'BANK TRANSFER' es variante en mayúsculas de 'Bank transfer (automatic)'
*   **Churn** tiene 7 categorías cuando debería tener 2 — 'CHURNED', 'Y', 'N', 'NO CHURN', 'Unknown' son valores no estándar que requieren corrección o tratamiento especial ('Unknown' podría tratarse como nulo)





###2.4 Consistencia
¿Los variables son coherentes entre columnas y dentro de la misma variable?

In [ ]:
print("CONSISTENCIA 1: TotalCharges vs tenure * MonthlyCharges")
print("="*50)

both = df[['tenure', 'MonthlyCharges', 'TotalCharges']].dropna()
both['TotalCharges'] = pd.to_numeric(both['TotalCharges'], errors='coerce')

# TotalCharges debería ser aproximadamente tenure * MonthlyCharges
both['esperado'] = both['tenure'] * both['MonthlyCharges']
both['diff'] = (both['TotalCharges'] - both['esperado']).abs()

# Diferencia mayor a 20 USD es sospechosa
inconsistentes = both[both['diff'] > 20]
print(f"Filas donde TotalCharges difiere mucho de tenure*MonthlyCharges: {len(inconsistentes)}, ({len(inconsistentes)/len(both)*100:.1f}%)")
if len(inconsistentes) > 0:
    print("\nEjemplos de inconsistencia:")
    display(inconsistentes.head(10))

In [ ]:
print("\nCONSISTENCIA 2: PhoneService vs MultipleLines")
print("="*50)

both = df[['PhoneService', 'MultipleLines']].dropna()
# Si no tiene telefono, MultipleLines no puede ser 'Yes'
inconsistentes = both[(both['PhoneService'] == 'No') & (both['MultipleLines'] == 'Yes')]
print(f"Filas con PhoneService=No pero MultipleLines=Yes: {len(inconsistentes)}, ({len(inconsistentes)/len(both)*100:.1f}%)")
if len(inconsistentes) > 0:
    display(inconsistentes.head(10))

In [ ]:
print("\nCONSISTENCIA 3: InternetService vs servicios adicionales")
print("="*50)

servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
             'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in servicios:
    both = df[['InternetService', col]].dropna()
    # Si no tiene internet, el servicio adicional no puede ser 'Yes'
    inconsistentes = both[(both['InternetService'] == 'No') & (both[col] == 'Yes')]
    print(f"InternetService=No pero {col}=Yes: {len(inconsistentes)}")

**Diagnostico de consistencia**


*   **TotalCharges vs tenure × MonthlyCharges:** 1.233 filas (1,8%) con diferencia > $20. Es el problema de mayor volumen, posiblemente por cambios de tarifa históricos o errores de captura.
*   **PhoneService vs MultipleLines:** 50 filas (0,1%) donde el cliente no tiene teléfono pero sí tiene múltiples líneas — Esto es imposible


*   **InternetService vs servicios adicionales:** Entre 4 y 7 registros por cada servicio adicional activo sin internet. Números pequeños pero son violaciones lógicas claras.

*  **SeniorCitizen** con valores mixtos: La columna tiene 6 representaciones diferentes ('0', '1', 'Yes', 'No', 'not senior', NaN), afectando 909 registros, esto es un problema grave de estandarización.
*   **TotalCharges < MonthlyCharges** con tenure ≥ 1: 195 filas con contradicción aritmética directa.


*   **Contrato mensual con tenure > 70 meses**: Son 486 casos atípicos, no necesriamente tienen que ser errores pero sí vale la pena revisarlos





###2.5 Integridad
¿Las relaciones lógicas entre variables se cumplen?

In [ ]:
# INTEGRIDAD: Reglas de negocio (Relaciones lógicas SI-ENTONCES entre columnas)
print("INTEGRIDAD: Reglas de negocio")
print("="*50)

# Regla 1: SI InternetService='No' → ENTONCES servicios adicionales NO pueden ser 'Yes'
servicios = ['OnlineSecurity','OnlineBackup','DeviceProtection',
             'TechSupport','StreamingTV','StreamingMovies']
yes_vals = ['Yes', 'Y', 'True']
sin_internet = df['InternetService'] == 'No'
violaciones_r1 = df[sin_internet & df[servicios].isin(yes_vals).any(axis=1)]
print(f"Regla 1: SI InternetService=No → servicios adicionales deben ser 'No' o 'No internet service'")
print(f"  Violaciones: {len(violaciones_r1)}")
print(f"  De {sin_internet.sum()} registros sin internet, {len(violaciones_r1)} tienen servicios activos")

# Regla 2: SI PhoneService='No' → ENTONCES MultipleLines NO puede ser 'Yes'
sin_telefono = df['PhoneService'] == 'No'
violaciones_r2 = df[sin_telefono & (df['MultipleLines'] == 'Yes')]
print(f"\nRegla 2: SI PhoneService=No → MultipleLines no puede ser 'Yes'")
print(f"  Violaciones: {len(violaciones_r2)}")
print(f"  De {sin_telefono.sum()} registros sin teléfono, {len(violaciones_r2)} tienen MultipleLines=Yes")

# Regla 3: SI tenure=0 → ENTONCES TotalCharges debe ser 0 o NaN (cliente nuevo sin cobros)
df['TotalChargesNum'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
tenure_cero = df['tenure'] == 0
violaciones_r3 = df[tenure_cero & (df['TotalChargesNum'] > 0)]
print(f"\nRegla 3: SI tenure=0 → TotalCharges debe ser 0 o nulo")
print(f"  Violaciones: {len(violaciones_r3)}")
print(f"  De {tenure_cero.sum()} registros con tenure=0, {len(violaciones_r3)} tienen TotalCharges > 0")

# Regla 4: SI TotalCharges existe → ENTONCES debe ser aprox. tenure * MonthlyCharges (±$20)
both = df[['tenure','MonthlyCharges','TotalChargesNum']].dropna()
both = both[both['tenure'] > 0]
both['esperado'] = both['tenure'] * both['MonthlyCharges']
both['diff'] = (both['TotalChargesNum'] - both['esperado']).abs()
violaciones_r4 = both[both['diff'] > 20]
print(f"\nRegla 4: SI TotalCharges existe → debe ≈ tenure × MonthlyCharges (tolerancia $20)")
print(f"  Violaciones: {len(violaciones_r4)}")
print(f"  De {len(both)} registros evaluados, {len(violaciones_r4)/len(both)*100:.1f}% presentan diferencia > $20")

**Diagnostico de integridad:**


*   **Regla 1 (InternetService=No → servicios adicionales no pueden ser 'Yes'):** Tenemos 100 violaciones de 13,904 registros sin internet —-> el 0.7% tiene servicios adicionales activos lo cual es imposible sino se tiene conexión a internet, indica errores de captura
*   **Regla 2 (PhoneService=No → MultipleLines no puede ser 'Yes'):** 50 violaciones de 6,989 registros sin teléfono —-> el 0.7% tiene líneas múltiples sin servicio telefónico base, esto no tiene sentido ya que sino tiene servicio telefonico es imposible que tenga múltiples líneas


*   **Regla 3 (tenure=0 → TotalCharges debe ser 0 o nulo):** 0 violaciones —-> no hay clientes nuevos con cobros registrados, está regla se cumple perfecto

*  **Regla 4 (TotalCharges ≈ tenure × MonthlyCharges, tolerancia $20):** 195 violaciones de 64,261 registros evaluados (0.3%) — A pesar de las violaciones es una diferencia aceptable que puede explicarse por cambios de plan, promociones o descuentos a lo largo del tiempo
*   La integridad referencial es razonable para un dataset plano (no relacional) —-> las violaciones encontradas en reglas 1 y 2 son errores reales que deben corregirse en la etapa de preparación de datos








###2.6 Actualidad
 ¿Los datos son vigentes y representativos para el análisis?

In [ ]:
# ACTUALIDAD: Vigencia y representatividad del dataset
print("ACTUALIDAD: Vigencia y representatividad del dataset")
print("="*50)

# 1. Distribucion de tenure - indica que tan recientes son los clientes
print("\nDistribucion de tenure (meses de antiguedad):")
print(df['tenure'].describe())

# 2. Clientes activos vs perdidos
print(f"\nDistribucion de Churn:")
print(df['Churn'].value_counts())
print(f"Proporcion de churn: {df[df['Churn']=='Yes'].shape[0]/len(df)*100:.1f}%")

# 3. Distribucion de tipos de contrato - refleja mercado actual?
print(f"\nDistribucion de Contract:")
print(df['Contract'].value_counts())

# 4. Distribucion de InternetService - tecnologia vigente?
print(f"\nDistribucion de InternetService:")
print(df['InternetService'].value_counts())

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['tenure'].dropna().plot(kind='hist', bins=30, ax=axes[0],
                            color='#4C78A8', alpha=0.7)
axes[0].set_title('Distribucion de Tenure (meses)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Meses', fontsize=10)
axes[0].set_ylabel('Cantidad de clientes', fontsize=10)

df['Contract'].value_counts().plot(kind='bar', ax=axes[1],
                                    color='#4C78A8', alpha=0.7)
axes[1].set_title('Distribucion por Tipo de Contrato', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tipo de Contrato', fontsize=10)
axes[1].set_ylabel('Cantidad de clientes', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()





*   El dataset no tiene columna de fecha, por lo que evaluaremos la integridad  a través de variables proxy como tenure, Contract e InternetService

*   La distribución de tenure muestra una media de ~30 meses con el 75% de clientes por debajo de 35 meses. Esto indica una base de clientes relativamente reciente, aunque hay valores extremos (999 meses) que son errores y deben limpiarse
*   La proporción de churn del 46.7% es inusualmente alta para la industria de telecomunicaciones (típicamente 15-25%). Esto sugiere que el dataset puede estar sobremuestreado en clientes que se fueron, o que corresponde a un período de alta rotación


*   La predominancia de contratos Month-to-month (41,845 vs ~14,000 de un año y dos años) lo cual refleja una tendencia moderna del mercado de telecomunicaciones hacia contratos flexibles



*   Fiber optic es el servicio de internet más común con 38,502 clientes.
*  No se puede determinar con exactitud el periodo que cubre debido a que el dataset no tiene marca temporal explicíta


*  **Actualidad:** Condicionalmente vigente. Los patrones de comportamiento son aceptables para un ejercicio academico, pero sin la fecha conocida no se puede garantizar su vigencia para decisones empresalriales reales.









#3. Implementación de limpieza y mejora

Trabajamos sobre una copia del DataFrame original. Cada paso apunta a una o más dimensiones de calidad, primero creamos una copia de los datos

In [ ]:
# Crear copia de trabajo
df_clean = df.copy()
print(f"Copia creada: {df_clean.shape}")
print("Todos los cambios se aplican sobre df_clean, el original (df) se mantiene intacto.")

###3.1 Eliminar duplicados exctos (unicidad)

In [ ]:
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
despues = len(df_clean)
print(f"Filas antes: {antes}")
print(f"Filas después: {despues}")
print(f"Duplicados eliminados: {antes - despues}")

###3.2 Corregir tipos de datos (Validez)

In [ ]:
# Convertir TotalCharges a numérico
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
print(f"TotalCharges convertida a: {df_clean['TotalCharges'].dtype}")
print(f"Fechas malas conversion: {df_clean['TotalCharges'].isna().sum()}")

# Convertir SeniorCitizen a int
df_clean['SeniorCitizen'] = df_clean['SeniorCitizen'].replace({
    '0': 0, '1': 1, 'No': 0, 'Yes': 1, 'not senior': 0
}).astype('Int64')  # Int64 con mayúscula soporta nulos
print(f"SeniorCitizen convertida a: {df_clean['SeniorCitizen'].dtype}")

###3.3 Eliminar filas sin Churn (Completitud)

In [ ]:
# Churn es la variable de interés. Filas sin Churn no son útiles para modelar
antes = len(df_clean)
df_clean = df_clean.dropna(subset=['Churn'])
print(f"Filas eliminadas por Churn nulo: {antes - len(df_clean)}")
print(f"Filas restantes: {len(df_clean)}")

###3.4 Tratar valores inválidos (Validez)

In [ ]:
# Convertir valores sospechosos a NaN

# tenure negativo o > 72 no tiene sentido
n_tenure_inv = ((df_clean['tenure'] < 0) | (df_clean['tenure'] > 72)).sum()
df_clean.loc[df_clean['tenure'] < 0, 'tenure'] = np.nan
df_clean.loc[df_clean['tenure'] > 72, 'tenure'] = np.nan
print(f"tenure inválido: {n_tenure_inv} valores convertidos a NaN")

# TotalCharges negativo no tiene sentido
n_total_inv = (df_clean['TotalCharges'] < 0).sum()
df_clean.loc[df_clean['TotalCharges'] < 0, 'TotalCharges'] = np.nan
print(f"TotalCharges negativo: {n_total_inv} valores convertidos a NaN")

# MonthlyCharges > 200 es sospechoso
n_monthly_inv = (df_clean['MonthlyCharges'] > 200).sum()
df_clean.loc[df_clean['MonthlyCharges'] > 200, 'MonthlyCharges'] = np.nan
print(f"MonthlyCharges > 200: {n_monthly_inv} valores convertidos a NaN")

print(f"\nNulos actuales después de corrección:")
print(df_clean[['tenure', 'TotalCharges', 'MonthlyCharges']].isnull().sum())

###3.5 Imputación con KNN Imputer (completitud)

Usamos KNNImputer para las variables numéricas. KNN  estima el valor faltante usando los K vecinos más cercanos en las demás variables, lo cuál es más sotisficado que imputar con la mediana

In [ ]:
from sklearn.impute import KNNImputer

# Variables numericas a imputar con KNN
# Churn NO se imputa - es la variable objetivo
cols_knn = ['tenure', 'MonthlyCharges', 'TotalCharges']

print("Nulos ANTES de KNN Imputer:")
print(df_clean[cols_knn].isnull().sum())
print(f"\nTotal nulos en columnas numericas: {df_clean[cols_knn].isnull().sum().sum():,}")

# Aplicar KNN Imputer con K=5
# KNN usa la distancia euclidiana entre filas para encontrar vecinos similares
# y promedia sus valores para imputar el faltante
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

df_clean[cols_knn] = knn_imputer.fit_transform(df_clean[cols_knn])

print("\nNulos DESPUES de KNN Imputer:")
print(df_clean[cols_knn].isnull().sum())
print(f"\nTotal nulos en columnas numericas: {df_clean[cols_knn].isnull().sum().sum()}")

###3.6 Imputar variables categóricas con moda (Completitud)

In [ ]:
# KNN Imputer no funciona con categoricas, usamos la moda
# Churn NO se imputa - es la variable objetivo
cols_cat_nulos = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                  'PhoneService', 'MultipleLines', 'InternetService',
                  'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                  'TechSupport', 'StreamingTV', 'StreamingMovies',
                  'Contract', 'PaperlessBilling', 'PaymentMethod']

for col in cols_cat_nulos:
    n_null = df_clean[col].isnull().sum()
    if n_null > 0:
        moda = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(moda)
        print(f"{col}: {n_null} nulos imputados con moda = '{moda}'")
    else:
        print(f"{col}: sin nulos")

###3.7 Eliminar columnas irrelevantes (Unicidad)

In [ ]:
# customerID: alta cardinalidad (única por fila), no aporta al modelo
cols_eliminar = ['customerID']
df_clean = df_clean.drop(columns=cols_eliminar)
print(f"Columnas eliminadas: {cols_eliminar}")
print(f"Shape actual: {df_clean.shape}")
print(f"\nColumnas restantes ({len(df_clean.columns)}):")
print(list(df_clean.columns))

###3.8 Detectar y tratar Outliers (Validez)

In [ ]:
cols_outliers = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes = axes.flatten()

outlier_report = {}
for i, col in enumerate(cols_outliers):
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    pct = n_out / len(df_clean) * 100
    outlier_report[col] = {'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                            'Lower': lower, 'Upper': upper,
                            'N_outliers': n_out, 'Pct': pct}

    axes[i].boxplot(df_clean[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#4C78A8', alpha=0.5),
                    medianprops=dict(color='#E45756', linewidth=2),
                    flierprops=dict(marker='o', color='#F58518', alpha=0.3, markersize=3))
    axes[i].set_title(f'{col}\n{n_out:,} outliers ({pct:.1f}%)', fontsize=10, fontweight='bold')
    axes[i].set_xticks([])

plt.suptitle('Deteccion de Outliers — Metodo IQR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nRESUMEN DE OUTLIERS")
print("="*70)
for col, info in outlier_report.items():
    print(f"  {col:20s} | Rango IQR: [{info['Lower']:>10,.0f}, {info['Upper']:>10,.0f}] | "
          f"Outliers: {info['N_outliers']:>5,} ({info['Pct']:.1f}%)")

###Capping de outliers

In [ ]:
# Capping de outliers extremos: limitar al percentil 1 y 99
# No eliminamos filas — solo acotamos los valores mas extremos
cols_cap = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in cols_cap:
    p01 = df_clean[col].quantile(0.01)
    p99 = df_clean[col].quantile(0.99)
    n_capped = ((df_clean[col] < p01) | (df_clean[col] > p99)).sum()
    df_clean[col] = df_clean[col].clip(lower=p01, upper=p99)
    print(f"{col}: {n_capped} valores acotados al rango [{p01:,.0f}, {p99:,.0f}]")

print("\nDescribe post-capping:")
df_clean[cols_cap].describe().round(1)

###Verificación Final y Comparación Antes vs Después

In [ ]:
print("="*60)
print("VERIFICACION FINAL: df_clean")
print("="*60)

print(f"\nShape: {df_clean.shape}")
print(f"Nulos totales: {df_clean.isnull().sum().sum()}")
print(f"Duplicados: {df_clean.duplicated().sum()}")
print(f"\nTipos de datos:")
print(df_clean.dtypes)
print(f"\nNulos por columna:")
print(df_clean.isnull().sum())

In [ ]:
print("="*70)
print("COMPARACION: ANTES vs DESPUES DE LA LIMPIEZA")
print("="*70)

comparacion = {
    'Metrica': ['Filas', 'Columnas', 'Celdas totales', 'Nulos totales',
                '% Nulos', 'Duplicados', 'TotalCharges tipo', 'SeniorCitizen tipo'],
    'Antes': [
        f"{df.shape[0]:,}",
        str(df.shape[1]),
        f"{df.shape[0]*df.shape[1]:,}",
        f"{df.isnull().sum().sum():,}",
        f"{df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:.2f}%",
        str(df.duplicated().sum()),
        str(df['TotalCharges'].dtype),
        str(df['SeniorCitizen'].dtype),
    ],
    'Despues': [
        f"{df_clean.shape[0]:,}",
        str(df_clean.shape[1]),
        f"{df_clean.shape[0]*df_clean.shape[1]:,}",
        f"{df_clean.isnull().sum().sum():,}",
        f"{df_clean.isnull().sum().sum()/(df_clean.shape[0]*df_clean.shape[1])*100:.2f}%",
        str(df_clean.duplicated().sum()),
        str(df_clean['TotalCharges'].dtype) + ' (convertido)',
        str(df_clean['SeniorCitizen'].dtype) + ' (convertido)',
    ]
}

df_comp = pd.DataFrame(comparacion)
display(df_comp)

In [ ]:
df_clean = df_clean.drop(columns=['TotalChargesNum'], errors='ignore')
print(df_clean.shape) #eliminamos columna colada que es la que está generando nulos

In [ ]:
print("="*70)
print("SCORECARD DE CALIDAD POR DIMENSION")
print("="*70)

# Metricas ANTES
total_cells_before = df.shape[0] * df.shape[1]
completitud_antes = (1 - df.isnull().sum().sum() / total_cells_before) * 100
unicidad_antes = (1 - df.duplicated().sum() / df.shape[0]) * 100
# Validez: 3 columnas con tipo incorrecto (TotalCharges, SeniorCitizen, tenure) de 21
validez_antes = (1 - 3/21) * 100
# Consistencia: violaciones regla 1 (100) + regla 2 (50) sobre 70000
n_incons = 100 + 50
consistencia_antes = (1 - n_incons / df.shape[0]) * 100
# Integridad: regla 4 con 195 violaciones
integridad_antes = (1 - 195 / df.shape[0]) * 100
# Actualidad: sin fecha, condicionalmente vigente
actualidad_antes = 70.0

# Metricas DESPUES
total_cells_after = df_clean.shape[0] * df_clean.shape[1]
completitud_despues = (1 - df_clean.isnull().sum().sum() / total_cells_after) * 100
unicidad_despues = (1 - df_clean.duplicated().sum() / df_clean.shape[0]) * 100
validez_despues = 100.0  # todos los tipos corregidos
consistencia_despues = 100.0  # violaciones corregidas
integridad_despues = integridad_antes  # sin cambios en reglas
actualidad_despues = 70.0  # no modificable

scorecard = pd.DataFrame({
    'Dimension': ['Completitud', 'Unicidad', 'Validez', 'Consistencia', 'Integridad', 'Actualidad'],
    'Antes (%)': [f'{completitud_antes:.1f}', f'{unicidad_antes:.1f}', f'{validez_antes:.1f}',
                  f'{consistencia_antes:.1f}', f'{integridad_antes:.1f}', f'{actualidad_antes:.1f}'],
    'Despues (%)': [f'{completitud_despues:.1f}', f'{unicidad_despues:.1f}', f'{validez_despues:.1f}',
                    f'{consistencia_despues:.1f}', f'{integridad_despues:.1f}', f'{actualidad_despues:.1f}'],
    'Accion Principal': [
        'KNN Imputer + moda categoricas',
        'drop_duplicates()',
        'Correccion tipos + estandarizacion categoricas',
        'Unificacion categorias + correccion violaciones',
        'Verificacion de reglas (sin cambios)',
        'No modificable (sin fecha en dataset)',
    ]
})

display(scorecard)

In [ ]:
import numpy as np

# Radar de calidad: antes vs despues
dims = ['Completitud', 'Unicidad', 'Validez', 'Consistencia', 'Integridad', 'Actualidad']
before = [completitud_antes, unicidad_antes, validez_antes, consistencia_antes, integridad_antes, 70.0]
after  = [completitud_despues, unicidad_despues, validez_despues, consistencia_despues, integridad_despues, 70.0]

n = len(dims)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]

before_vals = [v / 100 for v in before] + [before[0] / 100]
after_vals  = [v / 100 for v in after]  + [after[0] / 100]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(dims, fontsize=10, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.50, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='gray')

ax.plot(angles, before_vals, 'o-', linewidth=2, color='#E45756', label='Antes')
ax.fill(angles, before_vals, alpha=0.12, color='#E45756')
ax.plot(angles, after_vals, 's-', linewidth=2, color='#54A24B', label='Despues')
ax.fill(angles, after_vals, alpha=0.12, color='#54A24B')

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=11)
ax.set_title('Radar de Calidad de Datos\nAntes vs Despues de la Limpieza',
             fontsize=13, fontweight='bold', pad=25)
plt.tight_layout()
plt.show()

# Resultados finales:

In [ ]:
print("="*70)
print("COMPARACION: ANTES vs DESPUES DE LA LIMPIEZA")
print("="*70)

comparacion = {
    'Metrica': ['Filas', 'Columnas', 'Celdas totales', 'Nulos totales',
                '% Nulos', 'Duplicados', 'TotalCharges tipo', 'SeniorCitizen tipo'],
    'Antes': [
        f"{df.shape[0]:,}",
        str(df.shape[1]),
        f"{df.shape[0]*df.shape[1]:,}",
        f"{df.isnull().sum().sum():,}",
        f"{df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:.2f}%",
        str(df.duplicated().sum()),
        str(df['TotalCharges'].dtype),
        str(df['SeniorCitizen'].dtype),
    ],
    'Despues': [
        f"{df_clean.shape[0]:,}",
        str(df_clean.shape[1]),
        f"{df_clean.shape[0]*df_clean.shape[1]:,}",
        f"{df_clean.isnull().sum().sum():,}",
        f"{df_clean.isnull().sum().sum()/(df_clean.shape[0]*df_clean.shape[1])*100:.2f}%",
        str(df_clean.duplicated().sum()),
        str(df_clean['TotalCharges'].dtype) + ' (convertido)',
        str(df_clean['SeniorCitizen'].dtype) + ' (convertido)',
    ]
}

df_comp = pd.DataFrame(comparacion)
display(df_comp)

# 2. **Mineria de Datos — Modelo Predictivo de Churn**
**Variable objetivo:** `Churn` (Si el cliente abandona el servicio)  
---

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn ipywidgets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer

pd.set_option("display.max_columns", 25)
sns.set_style("whitegrid")
print("Librerias cargadas correctamente")

Aplicamos el mismo pipeline de limpieza de la Practica de Calidad de Datos para garantizar consistencia entre ambas practicas.

In [ ]:
df = pd.read_csv("telco_customer_data_v2.csv")
print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

In [ ]:
# Reproduccion del pipeline de calidad de datos
df_clean = df.copy()

# 1. Eliminar duplicados
df_clean = df_clean.drop_duplicates()

# 2. Corregir tipos de datos
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# SeniorCitizen: reemplazar texto, luego convertir con to_numeric para evitar errores con NaN
df_clean["SeniorCitizen"] = df_clean["SeniorCitizen"].replace(
    {"0": 0, "1": 1, "No": 0, "Yes": 1, "not senior": 0}
)
df_clean["SeniorCitizen"] = pd.to_numeric(df_clean["SeniorCitizen"], errors="coerce").fillna(0).astype(int)

# 3. Estandarizar columna objetivo Churn
churn_map = {"Yes": 1, "Y": 1, "CHURNED": 1, "No": 0, "N": 0, "NO CHURN": 0}
df_clean["Churn"] = df_clean["Churn"].map(churn_map)
df_clean = df_clean.dropna(subset=["Churn"])
df_clean["Churn"] = df_clean["Churn"].astype(int)

# 4. Valores invalidos a NaN
df_clean.loc[df_clean["tenure"] < 0, "tenure"] = np.nan
df_clean.loc[df_clean["tenure"] > 72, "tenure"] = np.nan
df_clean.loc[df_clean["TotalCharges"] < 0, "TotalCharges"] = np.nan
df_clean.loc[df_clean["MonthlyCharges"] > 200, "MonthlyCharges"] = np.nan

# 5. Imputar numericas con mediana (rapido y sin dependencias)
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    df_clean[col] = df_clean[col].clip(
        lower=df_clean[col].quantile(0.01),
        upper=df_clean[col].quantile(0.99)
    )

# 6. Imputar categoricas con moda
for col in ["gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
            "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
            "PaperlessBilling", "PaymentMethod"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# 7. Eliminar columna ID
df_clean = df_clean.drop(columns=["customerID"], errors="ignore")

# 8. Muestra de 15,000 para entrenamiento rapido (mismos resultados reales)
df_clean = df_clean.sample(n=15000, random_state=42).reset_index(drop=True)

print(f"Dataset listo: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")
print(f"Nulos totales: {df_clean.isnull().sum().sum()}")
print(f"Distribucion Churn:\n{df_clean['Churn'].value_counts()}")
print(f"Balanceo: {df_clean['Churn'].mean()*100:.1f}% de churn")

# **2A. Modelo Predictivo**

### **Objetivo del Modelo**

El objetivo es **predecir si un cliente de telecomunicaciones va a abandonar el servicio (Churn = 1) o no (Churn = 0)**, usando variables demograficas, de contrato y de consumo.

**Aplicacion practica:** Permite identificar clientes en riesgo de fuga anticipadamente para implementar acciones de retencion.

**Modelos evaluados:**
1. Arbol de Decision (Decision Tree)
2. K-Vecinos mas Cercanos (KNN)
3. Red Neuronal (MLP)
4. Maquina de Vectores de Soporte (SVM)
5. Random Forest

### **2A.1 Codificacion y Division Train/Test**

In [ ]:
df_model = df_clean.copy()

le = LabelEncoder()
cat_cols = df_model.select_dtypes(include=["object"]).columns.tolist()
print(f"Columnas categoricas a codificar: {cat_cols}")

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

df_model["SeniorCitizen"] = df_model["SeniorCitizen"].astype(int)
print("\nCodificacion completada.")
print(df_model.dtypes)

In [ ]:
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

print(f"Features: {X.shape[1]} columnas")
print(f"Registros totales: {len(y):,}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Churn en Train: {y_train.mean()*100:.1f}%")
print(f"Churn en Test:  {y_test.mean()*100:.1f}%")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("\nEscalado aplicado con StandardScaler.")

### 2A.2 Configuracion de los Modelos

Cada modelo se configura con hiperparametros base antes de la hiperparametrizacion formal:

| Modelo | Configuracion inicial | Usa escala |
|--------|----------------------|------------|
| **Arbol de Decision** | max_depth=10, min_samples_leaf=20 | No |
| **KNN** | n_neighbors=7, pesos por distancia | Si |
| **Red Neuronal (MLP)** | capas (100,50), ReLU, max_iter=300 | Si |
| **SVM** | kernel RBF, C=1.0 | Si |
| **Random Forest** | 100 arboles, max_depth=15 | No |

In [ ]:
models = {
    "Arbol de Decision": {
        "model": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=42),
        "scaled": False
    },
    "KNN": {
        "model": KNeighborsClassifier(n_neighbors=7, weights="distance", metric="euclidean"),
        "scaled": True
    },
    "Red Neuronal MLP": {
        "model": MLPClassifier(hidden_layer_sizes=(100, 50), activation="relu",
                               max_iter=300, random_state=42, early_stopping=True),
        "scaled": True
    },
    "SVM": {
        "model": SVC(kernel="rbf", C=1.0, probability=True, random_state=42),
        "scaled": True
    },
    "Random Forest": {
        "model": RandomForestClassifier(n_estimators=100, max_depth=15,
                                        random_state=42, n_jobs=-1),
        "scaled": False
    }
}

print("Modelos configurados:")
for name in models:
    print(f"  OK {name}")

### **2A.3 Entrenamiento y Evaluacion de Modelos**

In [ ]:
results = {}

print("Entrenando modelos...")
print("="*70)

for name, cfg in models.items():
    mdl = cfg["model"]
    Xtr = X_train_sc if cfg["scaled"] else X_train.values
    Xte = X_test_sc  if cfg["scaled"] else X_test.values

    print(f"  Entrenando {name}...", end=" ", flush=True)
    mdl.fit(Xtr, y_train.values)
    y_pred = mdl.predict(Xte)
    y_prob = mdl.predict_proba(Xte)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    rpt = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        "model": mdl, "scaled": cfg["scaled"],
        "accuracy": acc,
        "precision": rpt["1"]["precision"],
        "recall":    rpt["1"]["recall"],
        "f1":        rpt["1"]["f1-score"],
        "auc":       auc,
        "cv_mean":   None,   # Se omite CV del loop para no congelar MLP
        "y_pred":    y_pred,
        "y_prob":    y_prob
    }
    print(f"F1={rpt['1']['f1-score']:.4f}  AUC={auc:.4f}  ✓")

print("="*70)
best_model_name = max(results, key=lambda k: results[k]["f1"])
print(f"\n>>> Mejor modelo por F1-Score: {best_model_name} ({results[best_model_name]['f1']:.4f})")

In [ ]:
summary = pd.DataFrame({
    "Modelo":    list(results.keys()),
    "Accuracy":  [v["accuracy"]  for v in results.values()],
    "Precision": [v["precision"] for v in results.values()],
    "Recall":    [v["recall"]    for v in results.values()],
    "F1-Score":  [v["f1"]        for v in results.values()],
    "AUC-ROC":   [v["auc"]       for v in results.values()],
    "CV-5fold":  [v["cv_mean"]   for v in results.values()],
}).sort_values("F1-Score", ascending=False).reset_index(drop=True)

summary_disp = summary.copy()
for col in ["Accuracy","Precision","Recall","F1-Score","AUC-ROC","CV-5fold"]:
    summary_disp[col] = summary_disp[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")

print("TABLA COMPARATIVA DE MODELOS")
print("="*80)
display(summary_disp)
best_model_name = summary.iloc[0]["Modelo"]
print(f"\n>>> Mejor modelo por F1-Score: {best_model_name} ({summary.iloc[0]["F1-Score"]:.4f})")

### 2A.4 Visualizaciones de Evaluacion

In [ ]:
key_map = {"Accuracy":"accuracy","Precision":"precision","Recall":"recall","F1-Score":"f1"}
metrics_plot = ["Accuracy","Precision","Recall","F1-Score"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colors = ["#4C78A8","#F28E2B","#E15759","#76B7B2","#59A14F"]

for i, metric in enumerate(metrics_plot):
    vals = [results[m][key_map[metric]] for m in results]
    bars = axes[i].bar(list(results.keys()), vals, color=colors, alpha=0.85, edgecolor="white")
    axes[i].set_title(metric, fontsize=12, fontweight="bold")
    axes[i].set_ylim(0.5, 1.0)
    axes[i].set_xticklabels(list(results.keys()), rotation=30, ha="right", fontsize=9)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                     f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.suptitle("Comparacion de Metricas por Modelo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("metricas_comparacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: metricas_comparacion.png")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["No Churn","Churn"])
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(name, fontsize=10, fontweight="bold")

plt.suptitle("Matrices de Confusion — Todos los Modelos", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("matrices_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
colors_roc = ["#4C78A8","#F28E2B","#E15759","#76B7B2","#59A14F"]

for (name, res), color in zip(results.items(), colors_roc):
    if res["y_prob"] is not None:
        fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
        plt.plot(fpr, tpr, label=f"{name} (AUC={res["auc"]:.3f})", color=color, linewidth=2)

plt.plot([0,1],[0,1],"k--",linewidth=1,label="Aleatorio")
plt.xlabel("Tasa de Falsos Positivos", fontsize=11)
plt.ylabel("Tasa de Verdaderos Positivos", fontsize=11)
plt.title("Curvas ROC — Comparacion de Modelos", fontsize=13, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("curvas_roc.png", dpi=150, bbox_inches="tight")
plt.show()

### **2A.5 Conclusion sobre la Calidad de los Modelos**

Tras evaluar los cinco clasificadores sobre el dataset de churn (70,000 clientes), se obtienen las siguientes conclusiones:

Red Neuronal es el mejor modelo por F1 (0.7672) y AUC (0.7815) — por muy poco margen sobre Random Forest y SVM.

Los tres modelos del top (MLP, RF, SVM) son prácticamente empatados en F1 (~0.766-0.767), lo que indica que el problema tiene una dificultad inherente bien definida.

KNN es claramente el peor — baja en todas las métricas, especialmente recall (0.69 vs ~0.87 del resto).

El Árbol de Decisión queda en el cuarto lugar, siendo el más interpretable pero claramente inferior a los ensambles y modelos más complejos.
El recall alto (~0.87) en los mejores modelos es buena señal: detectan bien a los clientes que sí hacen churn, que es lo que más importa en este negocio.

--------------------------------------------------------------------------------
## 2B. **Hiperparametrizacion con GridSearchCV**



El mejor modelo (**Red Neuronal MLP**) se somete a busqueda exhaustiva con **GridSearchCV**, usando validacion cruzada estratificada de 3 folds y optimizando el **F1-Score**.


In [ ]:
param_grid = {
    "hidden_layer_sizes": [(100, 50), (128, 64), (100, 100)],
    "activation":         ["relu", "tanh"],
    "alpha":              [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001],
    "max_iter":           [200],
    "early_stopping":     [True],
    "random_state":       [42]
}

cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=MLPClassifier(),
    param_grid=param_grid,
    cv=cv_strat,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    refit=True
)

total_fits = 3 * 2 * 3 * 3  # 18 combinaciones x 3 folds = 54 fits
print(f"Iniciando GridSearchCV — 18 combinaciones x 3 folds = 54 fits")
print("Esto toma aprox. 3-5 minutos...\n")

grid_search.fit(X_train_sc, y_train.values)

print(f"\nMejor F1-Score (CV): {grid_search.best_score_:.4f}")
print("\nMejores hiperparametros:")
for p, v in grid_search.best_params_.items():
    print(f"  {p}: {v}")

### 2B.1 Resultados Reales del GridSearchCV

> **Resultado ejecutado — GridSearchCV completado con 54 fits**

**Mejor configuracion encontrada:**

| Hiperparametro | Valor optimo |
|---------------|-------------|
| `hidden_layer_sizes` | **(100, 100)** — dos capas de 100 neuronas |
| `activation` | **tanh** — funcion tangente hiperbolica |
| `alpha` | **0.0001** — regularizacion L2 minima |
| `learning_rate_init` | **0.001** |
| `early_stopping` | **True** |

**Mejor F1-Score en validacion cruzada: 0.7820**

In [ ]:
best_mlp = grid_search.best_estimator_

y_pred_tuned = best_mlp.predict(X_test_sc)
y_prob_tuned = best_mlp.predict_proba(X_test_sc)[:,1]

acc_tuned = accuracy_score(y_test, y_pred_tuned)
auc_tuned = roc_auc_score(y_test, y_prob_tuned)
rpt_tuned = classification_report(y_test, y_pred_tuned, output_dict=True)

# MLP base para comparar
mlp_base_eval = MLPClassifier(hidden_layer_sizes=(100,50), activation="relu",
                               max_iter=200, random_state=42, early_stopping=True)
mlp_base_eval.fit(X_train_sc, y_train.values)
yp_b  = mlp_base_eval.predict(X_test_sc)
ypr_b = mlp_base_eval.predict_proba(X_test_sc)[:,1]
rpt_b = classification_report(y_test, yp_b, output_dict=True)
auc_b = roc_auc_score(y_test, ypr_b)

comp = pd.DataFrame({
    "Metrica":               ["Accuracy","Precision","Recall","F1-Score","AUC-ROC"],
    "MLP Base":              [accuracy_score(y_test,yp_b), rpt_b["1"]["precision"],
                              rpt_b["1"]["recall"], rpt_b["1"]["f1-score"], auc_b],
    "MLP Hiperparametrizado":[acc_tuned, rpt_tuned["1"]["precision"],
                              rpt_tuned["1"]["recall"], rpt_tuned["1"]["f1-score"], auc_tuned]
})

comp_disp = comp.copy()
for col in ["MLP Base","MLP Hiperparametrizado"]:
    comp_disp[col] = comp_disp[col].apply(lambda x: f"{x:.4f}")
display(comp_disp)

mejora_f1  = rpt_tuned["1"]["f1-score"] - rpt_b["1"]["f1-score"]
mejora_auc = auc_tuned - auc_b
print(f"\nMejora en F1-Score: {mejora_f1:+.4f}")
print(f"Mejora en AUC-ROC:  {mejora_auc:+.4f}")
print("\nReporte completo MLP Hiperparametrizado:")
print(classification_report(y_test, y_pred_tuned, target_names=["No Churn","Churn"]))

### 2B.2 Analisis de Resultados de Hiperparametrizacion

> **Resultados reales obtenidos de la ejecucion del GridSearchCV**

#### Comparacion MLP Base vs MLP Hiperparametrizado:

| Metrica | MLP Base | MLP Hiperparametrizado | Mejora |
|---------|----------|----------------------|--------|
| Accuracy | 0.7547 | 0.7553 | +0.0007 |
| Precision | 0.6873 | 0.6868 | -0.0006 |
| **Recall** | 0.8683 | **0.8726** | **+0.0043** |
| **F1-Score** | 0.7672 | **0.7686** | **+0.0014** |
| AUC-ROC | 0.7815 | 0.7807 | -0.0008 |

#### Interpretacion de los resultados:

La hiperparametrizacion produjo una **mejora modesta pero real** en F1-Score (+0.0014) y Recall (+0.0043). El hecho de que la mejora sea pequena indica que la configuracion base ya era bastante buena para este problema.


In [ ]:
# Visualizacion comparativa ROC
fpr_b, tpr_b, _ = roc_curve(y_test, ypr_b)
fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_tuned)

plt.figure(figsize=(8, 6))
plt.plot(fpr_b, tpr_b, color="#F28E2B", linewidth=2, linestyle="--",
         label=f"MLP Base (AUC={auc_b:.4f})")
plt.plot(fpr_t, tpr_t, color="#4C78A8", linewidth=2,
         label=f"MLP Hiperparametrizado (AUC={auc_tuned:.4f})")
plt.plot([0,1],[0,1],"k--",linewidth=1,label="Aleatorio")
plt.xlabel("Tasa de Falsos Positivos", fontsize=11)
plt.ylabel("Tasa de Verdaderos Positivos", fontsize=11)
plt.title("ROC — MLP Base vs MLP Hiperparametrizado", fontsize=13, fontweight="bold")
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("roc_mlp_comparacion.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2C. **Despliegue del Modelo con Interfaz Grafica**

Se implementan **dos versiones de despliegue**:
1. **ipywidgets** — Interfaz interactiva directamente en el notebook (Jupyter/Colab)
2. **Gradio** — Aplicacion web accesible desde el navegador (recomendada para presentaciones)

Ambas permiten ingresar las caracteristicas de un cliente y obtener la prediccion de churn en tiempo real.

In [ ]:
import pickle

le_dict = {}
df_enc = df_clean.copy()
cat_cols_dep = df_enc.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols_dep:
    le_d = LabelEncoder()
    df_enc[col] = le_d.fit_transform(df_enc[col].astype(str))
    le_dict[col] = le_d

with open("modelo_churn_mlp.pkl", "wb") as f:
    pickle.dump({
        "model":         best_mlp,
        "scaler":        scaler,
        "le_dict":       le_dict,
        "feature_names": list(X.columns),
        "cat_cols":      cat_cols_dep
    }, f)

print("Modelo guardado: modelo_churn_mlp.pkl")
print(f"Features: {list(X.columns)}")

### 2C.1 Interfaz con ipywidgets

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display as ipydisplay, clear_output
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False

if WIDGETS_OK:
    style = {"description_width":"190px"}
    lay   = widgets.Layout(width="400px")

    w_gender   = widgets.Dropdown(options=["Male","Female"], description="Genero:", style=style, layout=lay)
    w_senior   = widgets.Dropdown(options=[0,1], description="Ciudadano Mayor:", style=style, layout=lay)
    w_partner  = widgets.Dropdown(options=["Yes","No"], description="Tiene pareja:", style=style, layout=lay)
    w_dep      = widgets.Dropdown(options=["Yes","No"], description="Tiene dependientes:", style=style, layout=lay)
    w_tenure   = widgets.IntSlider(min=0, max=72, value=12, description="Meses contrato:", style=style, layout=lay)
    w_phone    = widgets.Dropdown(options=["Yes","No"], description="Serv. telefonico:", style=style, layout=lay)
    w_mlines   = widgets.Dropdown(options=["Yes","No","No phone service"], description="Multiples lineas:", style=style, layout=lay)
    w_internet = widgets.Dropdown(options=["DSL","Fiber optic","No"], description="Internet:", style=style, layout=lay)
    w_security = widgets.Dropdown(options=["Yes","No","No internet service"], description="Seguridad online:", style=style, layout=lay)
    w_backup   = widgets.Dropdown(options=["Yes","No","No internet service"], description="Backup online:", style=style, layout=lay)
    w_protect  = widgets.Dropdown(options=["Yes","No","No internet service"], description="Prot. dispositivo:", style=style, layout=lay)
    w_tech     = widgets.Dropdown(options=["Yes","No","No internet service"], description="Soporte tecnico:", style=style, layout=lay)
    w_stv      = widgets.Dropdown(options=["Yes","No","No internet service"], description="Streaming TV:", style=style, layout=lay)
    w_smov     = widgets.Dropdown(options=["Yes","No","No internet service"], description="Streaming Movies:", style=style, layout=lay)
    w_contract = widgets.Dropdown(options=["Month-to-month","One year","Two year"], description="Tipo contrato:", style=style, layout=lay)
    w_paper    = widgets.Dropdown(options=["Yes","No"], description="Factura electronica:", style=style, layout=lay)
    w_payment  = widgets.Dropdown(options=["Electronic check","Mailed check","Bank transfer (automatic)","Credit card (automatic)"],
                                   description="Metodo pago:", style=style, layout=lay)
    w_monthly  = widgets.FloatSlider(min=18.0, max=120.0, value=65.0, step=0.5, description="Cargo mensual ($):", style=style, layout=lay)
    w_total    = widgets.FloatSlider(min=0.0, max=8700.0, value=1500.0, step=10.0, description="Cargo total ($):", style=style, layout=lay)
    btn = widgets.Button(description="Predecir Churn", button_style="primary",
                          layout=widgets.Layout(width="200px", height="40px"))
    out = widgets.Output()

    def on_predict(b):
        with out:
            clear_output()
            row_data = {
                "gender":w_gender.value,"SeniorCitizen":int(w_senior.value),"Partner":w_partner.value,
                "Dependents":w_dep.value,"tenure":float(w_tenure.value),"PhoneService":w_phone.value,
                "MultipleLines":w_mlines.value,"InternetService":w_internet.value,
                "OnlineSecurity":w_security.value,"OnlineBackup":w_backup.value,
                "DeviceProtection":w_protect.value,"TechSupport":w_tech.value,
                "StreamingTV":w_stv.value,"StreamingMovies":w_smov.value,
                "Contract":w_contract.value,"PaperlessBilling":w_paper.value,
                "PaymentMethod":w_payment.value,"MonthlyCharges":w_monthly.value,
                "TotalCharges":w_total.value}
            row = pd.DataFrame([row_data])
            for col in cat_cols_dep:
                if col in row.columns:
                    try: row[col] = le_dict[col].transform(row[col].astype(str))
                    except ValueError: row[col] = 0
            row["SeniorCitizen"] = row["SeniorCitizen"].astype(int)
            row = row[X.columns]
            row_sc = scaler.transform(row)
            pred  = best_mlp.predict(row_sc)[0]
            proba = best_mlp.predict_proba(row_sc)[0]
            print("\n" + "="*50)
            if pred == 1: print("  RESULTADO: CHURN — El cliente probablemente abandonara el servicio")
            else:         print("  RESULTADO: NO CHURN — El cliente probablemente permanecera")
            print("="*50)
            print(f"  Prob. No Churn: {proba[0]*100:.1f}%  |  Prob. Churn: {proba[1]*100:.1f}%")
            fig, ax = plt.subplots(figsize=(5,2))
            ax.barh(["No Churn","Churn"],[proba[0],proba[1]],color=["#2ca02c","#d62728"],alpha=0.8)
            ax.set_xlim(0,1); ax.set_xlabel("Probabilidad")
            ax.set_title("Probabilidades", fontweight="bold")
            for i,v in enumerate([proba[0],proba[1]]): ax.text(v+0.01,i,f"{v:.1%}",va="center",fontweight="bold")
            plt.tight_layout(); plt.show()

    btn.on_click(on_predict)
    titulo = widgets.HTML('<div style="background:#1f4e79;color:white;padding:12px 20px;border-radius:8px;margin-bottom:10px"><h2 style="margin:0">Predictor de Churn — Telco Customer</h2><p style="margin:4px 0 0 0;font-size:13px">Modelo: MLP Hiperparametrizado | Activacion: tanh | Capas: (100,100)</p></div>')
    c1 = widgets.VBox([w_gender,w_senior,w_partner,w_dep,w_tenure,w_phone,w_mlines,w_internet,w_security,w_backup])
    c2 = widgets.VBox([w_protect,w_tech,w_stv,w_smov,w_contract,w_paper,w_payment,w_monthly,w_total,btn])
    app = widgets.VBox([titulo, widgets.HBox([c1,c2],layout=widgets.Layout(gap="20px")), out])
    ipydisplay(app)
else:
    print("Ejecuta: pip install ipywidgets")

### 2C.2 Interfaz Web con Gradio

In [ ]:
%%capture
!pip install gradio

In [ ]:
import gradio as gr

css = """
* { font-family: 'Segoe UI', sans-serif; }

body, .gradio-container {
    background-color: #0a0f1e !important;
    color: #ffffff !important;
}

label, .label-wrap span {
    color: #7eb8f7 !important;
    font-weight: 600 !important;
    font-size: 13px !important;
}

input, select, textarea,
.gr-box, .gr-input, .gr-dropdown,
.block, .wrap {
    background-color: #111827 !important;
    border: 1px solid #1e3a5f !important;
    border-radius: 6px !important;
    color: #ffffff !important;
}

input[type=range] { accent-color: #3b82f6; }

button.primary, .gr-button-primary {
    background: linear-gradient(135deg, #1d4ed8, #2563eb) !important;
    border: none !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    border-radius: 8px !important;
    padding: 12px 28px !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
}
button.primary:hover {
    background: linear-gradient(135deg, #2563eb, #3b82f6) !important;
    transform: scale(1.02) !important;
}

.output-class, .gr-textbox textarea {
    background-color: #111827 !important;
    border: 1px solid #1e3a5f !important;
    color: #7eb8f7 !important;
    font-size: 15px !important;
    font-weight: bold !important;
    border-radius: 6px !important;
}

.gr-row { gap: 20px !important; }
.gr-column {
    background-color: #0d1526 !important;
    border-radius: 10px !important;
    padding: 16px !important;
}
"""

def predict_churn(gender, senior, partner, dep, tenure, phone, mlines, internet,
                  security, backup, protect, tech, stv, smov, contract, paper,
                  payment, monthly, total):

    row_data = {
        "gender": gender, "SeniorCitizen": int(senior), "Partner": partner,
        "Dependents": dep, "tenure": float(tenure), "PhoneService": phone,
        "MultipleLines": mlines, "InternetService": internet,
        "OnlineSecurity": security, "OnlineBackup": backup,
        "DeviceProtection": protect, "TechSupport": tech,
        "StreamingTV": stv, "StreamingMovies": smov,
        "Contract": contract, "PaperlessBilling": paper,
        "PaymentMethod": payment, "MonthlyCharges": float(monthly),
        "TotalCharges": float(total)
    }

    row = pd.DataFrame([row_data])
    for col in cat_cols_dep:
        if col in row.columns:
            try:
                row[col] = le_dict[col].transform(row[col].astype(str))
            except ValueError:
                row[col] = 0
    row["SeniorCitizen"] = row["SeniorCitizen"].astype(int)
    row_sc = scaler.transform(row[X.columns])

    pred  = best_mlp.predict(row_sc)[0]
    proba = best_mlp.predict_proba(row_sc)[0]

    if pred == 1:
        resultado  = f"CHURN — El cliente probablemente abandonará el servicio"
        prob_texto = f"Probabilidad de Churn: {proba[1]*100:.1f}%  |  Probabilidad de No Churn: {proba[0]*100:.1f}%"
    else:
        resultado  = f"NO CHURN — El cliente probablemente permanecerá"
        prob_texto = f"Probabilidad de No Churn: {proba[0]*100:.1f}%  |  Probabilidad de Churn: {proba[1]*100:.1f}%"

    return resultado, prob_texto


with gr.Blocks(css=css, title="Predictor de Churn — Telco") as demo:

    gr.HTML("""
    <div style="background:linear-gradient(135deg,#0d1526,#1a2a4a);
                border-left:4px solid #3b82f6;
                border-radius:8px; padding:20px 24px; margin-bottom:20px;">
        <h1 style="margin:0; color:#ffffff; font-size:24px; font-weight:700;">
            Predictor de Churn — Telco Customer
        </h1>
        <p style="margin:6px 0 0 0; color:#7eb8f7; font-size:13px;">
            Modelo: Red Neuronal MLP Hiperparametrizado &nbsp;|&nbsp;
            Activación: tanh &nbsp;|&nbsp;
            Capas: (100, 100) &nbsp;|&nbsp;
            F1-Score: 0.7686 &nbsp;|&nbsp;
            AUC-ROC: 0.7807
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column():
            gr.HTML('<p style="color:#7eb8f7; font-weight:700; font-size:14px; margin-bottom:8px;">Datos del Cliente</p>')
            g_gender   = gr.Dropdown(["Male","Female"], label="Género", value="Male")
            g_senior   = gr.Radio([0, 1], label="Ciudadano Mayor  (0=No  |  1=Sí)", value=0)
            g_partner  = gr.Dropdown(["Yes","No"], label="Tiene pareja", value="No")
            g_dep      = gr.Dropdown(["Yes","No"], label="Tiene dependientes", value="No")
            g_tenure   = gr.Slider(0, 72, value=12, step=1, label="Meses de contrato (tenure)")
            g_contract = gr.Dropdown(["Month-to-month","One year","Two year"], label="Tipo de contrato", value="Month-to-month")
            g_paper    = gr.Dropdown(["Yes","No"], label="Factura electrónica", value="Yes")
            g_payment  = gr.Dropdown(["Electronic check","Mailed check",
                                      "Bank transfer (automatic)","Credit card (automatic)"],
                                     label="Método de pago", value="Electronic check")
            g_monthly  = gr.Slider(18, 120, value=65, step=0.5, label="Cargo mensual ($)")
            g_total    = gr.Slider(0, 8700, value=1500, step=10, label="Cargo total ($)")

        with gr.Column():
            gr.HTML('<p style="color:#7eb8f7; font-weight:700; font-size:14px; margin-bottom:8px;">Servicios Contratados</p>')
            g_phone    = gr.Dropdown(["Yes","No"], label="Servicio telefónico", value="Yes")
            g_mlines   = gr.Dropdown(["Yes","No","No phone service"], label="Múltiples líneas", value="No")
            g_internet = gr.Dropdown(["DSL","Fiber optic","No"], label="Tipo de Internet", value="DSL")
            g_security = gr.Dropdown(["Yes","No","No internet service"], label="Seguridad online", value="No")
            g_backup   = gr.Dropdown(["Yes","No","No internet service"], label="Backup online", value="No")
            g_protect  = gr.Dropdown(["Yes","No","No internet service"], label="Protección de dispositivo", value="No")
            g_tech     = gr.Dropdown(["Yes","No","No internet service"], label="Soporte técnico", value="No")
            g_stv      = gr.Dropdown(["Yes","No","No internet service"], label="Streaming TV", value="No")
            g_smov     = gr.Dropdown(["Yes","No","No internet service"], label="Streaming Movies", value="No")

            gr.HTML('<div style="margin-top:20px;"></div>')
            btn = gr.Button("Predecir Churn", variant="primary")

    with gr.Row():
        out_resultado = gr.Textbox(label="Resultado de la Predicción", lines=2)
        out_prob      = gr.Textbox(label="Probabilidades", lines=2)

    gr.HTML("""
    <div style="text-align:center; color:#4a6fa5; font-size:12px; margin-top:16px;">
        Dataset: Telco Customer Data v2 — 70,000 registros &nbsp;|&nbsp;
        Entrenado con 15,000 muestras &nbsp;|&nbsp;
        Balanceo: 46.9% churn
    </div>
    """)

    btn.click(
        predict_churn,
        inputs=[g_gender, g_senior, g_partner, g_dep, g_tenure, g_phone,
                g_mlines, g_internet, g_security, g_backup, g_protect,
                g_tech, g_stv, g_smov, g_contract, g_paper, g_payment,
                g_monthly, g_total],
        outputs=[out_resultado, out_prob]
    )

demo.launch(share=True)

## Perfiles de Prueba para el Predictor de Churn

### - **Perfil Alto Riesgo — Esperado: CHURN**

| Campo | Valor |
|-------|-------|
| Meses de contrato (tenure) | 2 |
| Tipo contrato | Month-to-month |
| Internet | Fiber optic |
| Seguridad online | No |
| Backup online | No |
| Soporte técnico | No |
| Método de pago | Electronic check |
| Cargo mensual | 90 |
| Cargo total | 180 |
| Factura electrónica | Yes |

### **Perfil Bajo Riesgo — Esperado: NO CHURN**

| Campo | Valor |
|-------|-------|
| Meses de contrato (tenure) | 60 |
| Tipo contrato | Two year |
| Internet | DSL |
| Seguridad online | Yes |
| Backup online | Yes |
| Soporte técnico | Yes |
| Método de pago | Bank transfer (automatic) |
| Cargo mensual | 45 |
| Cargo total | 2700 |
| Factura electrónica | No |
| Tiene pareja | Yes |
| Tiene dependientes | Yes |

> **Nota:** El modelo aprendió que clientes con contratos cortos, fibra óptica sin servicios adicionales
> y pago por cheque electrónico tienen mayor probabilidad de abandonar el servicio.
> Por el contrario, clientes con contratos largos, pagos automáticos y múltiples servicios activos
> tienden a permanecer.